# Nhận diện MSSV qua Camera (InsightFace + FAISS)

Notebook này dùng webcam để nhận diện sinh viên theo thời gian thực.

**Yêu cầu:** Đã chạy notebook `face_recognition_insightface_faiss.ipynb` để tạo `models/faiss.index` và `models/id_map.json`.

**Tối ưu:** GPU (CUDA) + nhận diện mỗi **5 frame** (`DETECT_EVERY`).

**Điều khiển:**
- Video ở cửa sổ OpenCV | **Q** / **ESC** hoặc nút **Dừng camera**
- Chạy **Restart Kernel** → cell 1 → 2 → 3 → 4 (cell 2 phải in `CUDAExecutionProvider`)

In [1]:
import json
import sys
import warnings
from pathlib import Path

# Cài ipywidgets nếu kernel chưa có (cần cho nút Dừng/Chụp trong notebook)
try:
    import ipywidgets  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets', 'ipympl', '-q'])

import cv2
import faiss
import numpy as np
from insightface.app import FaceAnalysis

warnings.filterwarnings('ignore')

# ─── CẤU HÌNH ───────────────────────────────────────────────────────────────
BASE_DIR   = Path('.').resolve()
MODEL_DIR  = BASE_DIR / 'models'
INDEX_PATH = MODEL_DIR / 'faiss.index'
IDMAP_PATH = MODEL_DIR / 'id_map.json'

THRESHOLD  = 0.60
K_NEIGHBORS = 5
DIM        = 512
DET_SIZE      = (640, 640)
DETECT_EVERY  = 5         # cứ N frame mới gọi InsightFace 1 lần
ONNX_PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider']

CAMERA_INDEX = 0          # đổi thành 1 nếu dùng camera ngoài
MAX_FRAMES   = None       # ví dụ 3000 để tự dừng; None = chạy đến khi bấm Dừng

assert INDEX_PATH.exists(), f'Không tìm thấy {INDEX_PATH}. Hãy train trước!'
assert IDMAP_PATH.exists(), f'Không tìm thấy {IDMAP_PATH}. Hãy train trước!'

print(f'Index : {INDEX_PATH.resolve()}')
print(f'Ngưỡng: {THRESHOLD} | Nhận diện mỗi {DETECT_EVERY} frame')

Index : D:\1_HOC\KLTN\NhanDangMSSV\models\faiss.index
Ngưỡng: 0.6 | Nhận diện mỗi 5 frame


In [2]:
# Load InsightFace + FAISS (GPU + chỉ detection + recognition)
import onnxruntime as ort

def _cuda_works() -> bool:
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        return False
    det_onnx = Path.home() / '.insightface/models/buffalo_l/det_10g.onnx'
    if not det_onnx.exists():
        return False
    try:
        ort.InferenceSession(str(det_onnx), providers=['CUDAExecutionProvider'])
        return True
    except Exception:
        return False

_use_cuda = _cuda_works()
_providers = ONNX_PROVIDERS if _use_cuda else ['CPUExecutionProvider']
if not _use_cuda:
    print('⚠ CUDA khong chay duoc (thieu CUDA/cuBLAS?) — dung CPU. Cai CUDA Toolkit + onnxruntime-gpu.')

face_app = FaceAnalysis(
    name='buffalo_l',
    allowed_modules=['detection', 'recognition'],
    providers=_providers,
)
face_app.prepare(ctx_id=0 if _use_cuda else -1, det_size=DET_SIZE)

faiss_index = faiss.read_index(str(INDEX_PATH))
with open(IDMAP_PATH, 'r', encoding='utf-8') as f:
    id_map = {int(k): v for k, v in json.load(f).items()}


def predict_from_embedding(embedding):
    """Trả về (mssv, conf) từ embedding bằng FAISS."""
    if embedding is None:
        return 'Unknown', 0.0
    emb = np.asarray(embedding, dtype='float32').reshape(1, -1)
    D, I = faiss_index.search(emb, 1)
    if I.size == 0 or I[0, 0] < 0:
        return 'Unknown', float(D[0, 0]) if D.size else 0.0
    idx = int(I[0, 0])
    conf = float(D[0, 0])
    mssv = id_map.get(idx, 'Unknown')
    if conf < THRESHOLD:
        return 'Unknown', conf
    return mssv, conf

print(f'FAISS: {faiss_index.ntotal} vectors | Providers: {_providers}')

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: C:\Users\Anh Kien/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: C:\Users\Anh Kien/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Anh Kien/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: C:\Users\Anh Kien/.insightface\models\buffalo_l\genderage.onnx genderage
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Anh Kien/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
FAISS: 606 vectors | Providers: ['

In [10]:
import sys
import time
import subprocess
from IPython.display import display, Image

try:
    import ipywidgets as widgets
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '-q'])
    import ipywidgets as widgets

WINDOW_NAME = 'Nhan dien MSSV'
state = {'stop': False, 'snap': False}
frame_count = 0
mirror_video = True
max_runtime_seconds = 120
max_frames = MAX_FRAMES if MAX_FRAMES is not None else 600
display_handle = None
camera_start_time = None


def _reset_camera_state():
    state['stop'] = False
    state['snap'] = False


def _safe_release(cap_obj):
    try:
        cap_obj.release()
    except Exception:
        pass
    try:
        cv2.destroyAllWindows()
    except Exception:
        pass


def _opencv_has_gui() -> bool:
    try:
        cv2.namedWindow('__gui_test__', cv2.WINDOW_NORMAL)
        cv2.destroyWindow('__gui_test__')
        return True
    except cv2.error:
        return False


USE_CV_WINDOW = _opencv_has_gui()

btn_stop = widgets.Button(description='Dung camera', button_style='danger')
btn_snap = widgets.Button(description='Chup man hinh', button_style='info')
btn_stop.on_click(lambda _: state.update({'stop': True}))
btn_snap.on_click(lambda _: state.update({'snap': True}))
video_out = widgets.Output()
display(widgets.VBox([widgets.HBox([btn_stop, btn_snap]), video_out]))

cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError(f'Khong mo duoc camera index={CAMERA_INDEX}')
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

_last_results = []


def draw_face_label(frame, bbox, mssv, conf):
    """Ve khung va nhan nhan dien len frame."""
    if bbox is None:
        return frame
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = max(x1 + 1, x2)
    y2 = max(y1 + 1, y2)
    color = (0, 200, 0) if mssv != 'Unknown' else (0, 165, 255)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    label = f'{mssv} | {conf:.2f}'
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    ty = max(0, y1 - th - 10)
    cv2.rectangle(frame, (x1, ty), (x1 + tw + 10, ty + th + 10), color, -1)
    cv2.putText(frame, label, (x1 + 5, ty + th + 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)
    return frame


def _detect_and_recognize(frame):
    global _last_results
    faces = face_app.get(frame, max_num=1)
    _last_results = []
    for face in faces:
        emb = face.normed_embedding.astype('float32')
        mssv, conf = predict_from_embedding(emb)
        _last_results.append({'bbox': face.bbox.copy(), 'mssv': mssv, 'conf': conf})


def _draw_cached(frame):
    if _last_results:
        for r in _last_results:
            draw_face_label(frame, r['bbox'], r['mssv'], r['conf'])
    else:
        cv2.putText(frame, 'Dua khuon mat vao khung hinh', (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 200, 255), 2, cv2.LINE_AA)
    return frame


def _show_frame(frame):
    global display_handle
    ok_enc, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
    if not ok_enc:
        return
    img = Image(data=buf.tobytes())
    if display_handle is None:
        display_handle = display(img, display_id=True)
    else:
        display_handle.update(img)


def _camera_loop():
    global frame_count, camera_start_time
    camera_start_time = time.time()
    while not state['stop']:
        if time.time() - camera_start_time > max_runtime_seconds:
            print('Het thoi gian cho phep, tu dong dung de tranh treo.')
            break
        ok, frame = cap.read()
        if not ok:
            print('Khong doc duoc frame tu camera.')
            break

        if mirror_video:
            frame = cv2.flip(frame, 1)

        if frame_count % DETECT_EVERY == 0:
            _detect_and_recognize(frame)
        _draw_cached(frame)

        if state['snap']:
            out = BASE_DIR / 'screenshot_camera.jpg'
            cv2.imwrite(str(out), frame)
            print(f'Da luu: {out.resolve()}')
            state['snap'] = False

        if USE_CV_WINDOW:
            cv2.imshow(WINDOW_NAME, frame)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), ord('Q'), 27):
                state['stop'] = True
        else:
            _show_frame(frame)
            time.sleep(0.01)

        frame_count += 1
        if frame_count >= max_frames:
            print(f'Da chay {max_frames} frame, tu dong dung.')
            break


print(f'Video: "{WINDOW_NAME}" | Nhan dien moi {DETECT_EVERY} frame | Soi guong = {mirror_video} | Q/ESC/Dung camera de thoat.')


def _run_and_cleanup():
    try:
        _reset_camera_state()
        _camera_loop()
    except KeyboardInterrupt:
        print('Da dung (Ctrl+C).')
    finally:
        state['stop'] = True
        _safe_release(cap)
        print('Da dung camera. Neu cell bi treo, chay Restart Kernel.')


# Chay camera o cell 4 de notebook co the khoi tao tu dau roi moi bat camera.


Video: "Nhan dien MSSV" | Nhan dien moi 5 frame | Soi guong = True | Q/ESC/Dung camera de thoat.


In [12]:
# Cell chay camera: sau khi chay cell 1 -> 2 -> 3, chay cell nay de bat camera.
if 'draw_face_label' not in globals():
    raise NameError('draw_face_label not defined. Hay Restart Kernel va chay lai cell 1 -> 2 -> 3 -> 4.')
_run_and_cleanup()

Khong doc duoc frame tu camera.
Da dung camera. Neu cell bi treo, chay Restart Kernel.
